# 并行中断

In [2]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt
from rich import print

# 声明状态
class State(TypedDict):
    username: str
    age: str


# 声明节点
def node_a(state: State) -> dict:
    username = interrupt("请输入你的名字")
    return {
        "username": username,
    }


def node_b(state: State) -> dict:
    age = interrupt("请输入你的年龄")
    return {
        "age": age,
    }


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)

{
    '__interrupt__': [
        Interrupt(value='请输入你的名字', id='b1c3ecaa7ba0290cb1ab968d8ce8ade8'),
        Interrupt(value='请输入你的年龄', id='fc57cd47f19ed7a4c21d8b763477302e')
    ]
}

In [ ]:
# 恢复执行
resume_map = {}  # { id : msg }
for i in res['__interrupt__']:
    ask_msg = input(f"{i.value}")
    resume_map[i.id] = ask_msg
resume_res = graph.invoke(Command(resume=resume_map), config=config)
print(resume_res)